爬取网站代码

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.getenv("token")

In [7]:
import requests
import time
import json

# ===== 基本配置 =====

headers = {
    "User-Agent": "Mozilla/5.0",
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
    "Origin": "https://ssemarket.cn",
    "Referer": "https://ssemarket.cn/new/"
}
# 在全局创建一个 session
session = requests.Session()
session.headers.update(headers)

# ===== 1️⃣ 获取帖子列表 =====
def get_post_list(page):
    url = "https://ssemarket.cn/api/auth/browse"

    data = {
        "limit": 20,
        "offset": page * 20,
        "partition": "主页",
        "searchsort": "home",
        "searchinfo": "",
        "tag": "",
        "userTelephone": "39396392616"
    }

    resp = session.post(url, json=data) 
    #resp 是<class 'requests.models.Response'>
    #resp.json()是列表,resp.json()是字典
    

   

    return resp.json()


# ===== 2️⃣ 获取帖子详情 =====
def get_post_detail(post_id):
    url = "https://ssemarket.cn/api/auth/showDetails"  

    data = {
        "postID": post_id,
        "postType": "post",
        "userTelephone": "39396392616"
    }

    resp = session.post(url, json=data)
    return resp.json()


# ===== 3️⃣ 获取评论 =====
def get_comments(post_id):
    url = "https://ssemarket.cn/api/auth/showPcomments"  

    comments = []

    for page in range(0, 5):
        data = {
            "postID": post_id,
            "postType": "post",
            "userTelephone": "39396392616",
            "offset": page * 10,
            "limit": 10
        }

        resp = session.post(url, json=data)
        result = resp.json()

        comment_list = result

        if not comment_list:
            break

        comments.extend(comment_list)

        

    return comments


# ===== 主流程 =====
def main():
    all_data = []
    page=0
    while True:
        result = get_post_list(page)

        posts = result  

        if not posts or len(posts) == 0:
            print("所有帖子已抓取完毕！")
            break

        for post in posts:
            if page < 1:
                print("原始post:", post)

            post_id = post.get("PostID")  

            if not post_id:
                continue

            print(f"正在抓取帖子: {post_id}")

            detail = get_post_detail(post_id)
            comments = get_comments(post_id)

            all_data.append({
                "post_id": post_id,
                "detail": detail,
                "comments": comments
            })
        page+=1
            

    



测试阶段只爬取十页,帖子数为47

第一版初始代码用时9m52.4s

第二版去掉time.sleep()用时3m32.7s

第三版用session替换requesets.post,因为每次调用都会产生一次TCP握手,用时25s; 把limit设为网站的最大值20会更快,约为11s


实践阶段,共4580个帖子
第一版用session,时间为15m16.9s


In [8]:
if __name__ == "__main__":
    main()

原始post: {'PostID': 4591, 'UserName': 'iop', 'UserScore': 239, 'UserTelephone': '65776306681', 'UserAvatar': '', 'UserIdentity': 'student', 'Title': '校园集市上看到的，我有相同疑问', 'Content': '“软工求助！\n请问一下软工25级等级制相关问题～\nrt，想问一下各位uu软院现在25级绩点制的话是不是会有很多人有相同的排名呀，那这样的话综测和推免要怎么算捏？\n感谢各位uu！”\n', 'Like': 0, 'Comment': 1, 'Browse': 352, 'Heat': 149, 'PostTime': '2026-03-26T12:29:53+08:00', 'IsSaved': False, 'IsLiked': False, 'Photos': '', 'Tag': ''}
正在抓取帖子: 4591
原始post: {'PostID': 4590, 'UserName': '超级小爱', 'UserScore': 1059, 'UserTelephone': '33869240661', 'UserAvatar': 'https://sse-market-source-1320172928.cos.ap-guangzhou.myqcloud.com/src/images/resized/1764900068325510301_avatar_1764900068206.png', 'UserIdentity': 'student', 'Title': '【开发招募】软件工程学院本科生综合素质测评系统', 'Content': '开发需求：\n目标：开发程序（或多维表格），通过在程序系统导入和填报数据（表格），由系统自动完成学院本科生每学年综合素质测评工作。\n\n已有数据：历年学院本科生综合素质测评数据，包含学生参评个人基础信息、学年学业平均绩点、学生综测加分等表格，学年评选结果（人工评选的标准答案） ，学院本科生综合素质测评实施方案 以及 评选规则标准（用于测试程序逻辑是否正确）等。\n\n要求：在用往年数据测试程序逻辑正确的基础上，导入和填报 今年 及 后续年份的学院本科生综合素质测评数据